In [ ]:
%pip install monai

from google.colab import drive
drive.mount('/content/drive')

import time, shutil, importlib, sys, os, json

import torch
import torch.nn as nn
import numpy as np
from monai.utils import set_determinism

ruta_tfm = '/content/drive/MyDrive/TFM'
if ruta_tfm not in sys.path:
    sys.path.append(ruta_tfm)

from data_loader import crear_dataloaders
from segnet import SegNet
import importlib, utils_training
importlib.reload(utils_training)
from utils_training import train_network

# 1. Definir el modelo específico del cuaderno
model = SegNet(in_channels=3, out_channels=1, channels=(16, 32, 64, 128, 256))

# 2. Definir el optimizador estandarizado
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-5)

DEVICE = "cuda" #if torch.cuda.is_available() else "cpu"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 29.0 MB/s eta 0:00:00
Mounted at /content/drive


In [ ]:
importlib.reload(utils_training)

set_determinism(seed=42)          # misma semilla en las 4 arquitecturas

# ---- 0. Comprobar que Drive sigue montado (si no, remontar y volver a ejecutar) ----
Path_JSON = "/content/drive/MyDrive/TFM/ISLES-2022_Preprocessed_v2/dataset_split.json"   # v2: rejilla alineada
Dir_Drive = "/content/drive/MyDrive/TFM/ISLES-2022_Preprocessed_v2"
Dir_NPY   = "/content/npy_local_v2"   # carpeta NUEVA: no reutiliza las copias desalineadas de la versión anterior

if not os.path.exists(Path_JSON):
    raise RuntimeError("Drive no está accesible. Ejecuta:\n"
                       "  from google.colab import drive; drive.mount('/content/drive', force_remount=True)\n"
                       "y vuelve a ejecutar esta celda.")

for viejo in ("/content/dataset_local", "/content/npy_local"):   # libera disco de intentos anteriores
    shutil.rmtree(viejo, ignore_errors=True)

# ---- 1. Partición fija (JSON) ----
with open(Path_JSON, "r") as f:
    dataset_split = json.load(f)

def id_base(s):
    # El JSON guarda 'sub-strokecase0062_FLAIR_proc.npy' (por el .split('_flair') en minúscula del
    # preprocesamiento). Esto lo normaliza a 'sub-strokecase0062' y también funciona si algún día
    # el JSON trae solo el ID.
    return s.split("_FLAIR")[0]

ids_train = [id_base(s) for s in dataset_split["train"]]
ids_val   = [id_base(s) for s in dataset_split["val"]]
print(f"✅ Partición: Train={len(ids_train)} | Val={len(ids_val)}  (el test NO se copia todavía)")

# ---- 2. Copia Drive -> SSD local: idempotente, con reintentos y formato compacto ----
# Imágenes -> float16 (el loader las convierte a float32); máscara -> uint8.
SUFIJOS = {"_FLAIR_proc.npy": np.float16, "_ADC_proc.npy": np.float16,
           "_dwi_proc.npy": np.float16,   "_msk_proc.npy": np.uint8}

def copiar_compacto(src, dst, dtype, intentos=3):
    if os.path.exists(dst):                       # ya copiado correctamente
        return True
    tmp = dst + ".tmp.npy"
    for k in range(intentos):
        try:
            arr = np.load(src)
            arr = (arr > 0.5) if dtype == np.uint8 else arr
            np.save(tmp, arr.astype(dtype))
            os.replace(tmp, dst)                  # atómico: nunca queda un archivo a medias
            return True
        except (OSError, ValueError) as e:
            print(f"  ⚠️ intento {k+1}/{intentos} falló en {os.path.basename(src)}: {e}")
            time.sleep(5)
    return False

os.makedirs(Dir_NPY, exist_ok=True)
todos = ids_train + ids_val
fallidos, t0 = [], time.time()
for i, sid in enumerate(todos, 1):
    for suf, dt in SUFIJOS.items():
        ok = copiar_compacto(os.path.join(Dir_Drive, sid + suf), os.path.join(Dir_NPY, sid + suf), dt)
        if not ok:
            fallidos.append(sid + suf)
    if i % 20 == 0 or i == len(todos):
        libre = shutil.disk_usage("/content").free / 1e9
        print(f"  {i}/{len(todos)} sujetos | {time.time()-t0:.0f}s | disco libre {libre:.1f} GB")
    if len(fallidos) >= 8:                        # el mount se cayó: no seguir quemando archivos
        break

if fallidos:
    raise RuntimeError(f"No se pudieron copiar {len(fallidos)} archivos (ej.: {fallidos[:3]}).\n"
                       "Remonta Drive con force_remount=True y vuelve a ejecutar: la celda continúa donde quedó.")
print("✅ Copia completa y verificada.")

# ---- 3. Rutas + dataloaders ----
archivos_train = [os.path.join(Dir_NPY, f"{s}_FLAIR_proc.npy") for s in ids_train]
archivos_val   = [os.path.join(Dir_NPY, f"{s}_FLAIR_proc.npy") for s in ids_val]

ruta_tfm = "/content/drive/MyDrive/TFM"
if ruta_tfm not in sys.path:
    sys.path.append(ruta_tfm)
import data_loader
importlib.reload(data_loader)                     # por si ya estaba importada la versión anterior
from data_loader import crear_dataloaders

train_loader, val_loader = crear_dataloaders(archivos_train, archivos_val,
                                             batch_size=16, num_workers=2,
                                             prop_sin_lesion_train=0.3)


✅ Partición: Train=160 | Val=40  (el test NO se copia todavía)
  20/200 sujetos | 45s | disco libre 199.5 GB
  40/200 sujetos | 89s | disco libre 196.9 GB
  60/200 sujetos | 134s | disco libre 194.3 GB
  80/200 sujetos | 177s | disco libre 191.6 GB
  100/200 sujetos | 220s | disco libre 189.0 GB
  120/200 sujetos | 264s | disco libre 186.4 GB
  140/200 sujetos | 307s | disco libre 183.8 GB
  160/200 sujetos | 351s | disco libre 181.2 GB
  180/200 sujetos | 393s | disco libre 178.6 GB
  200/200 sujetos | 436s | disco libre 176.0 GB
✅ Copia completa y verificada.
[Dataset] sujetos=160 | cortes con lesión=7541 | sin lesión (con cerebro)=13714 -> usados=4116 | total=11657
[Dataset] sujetos=40 | cortes con lesión=1888 | sin lesión (con cerebro)=3426 -> usados=3426 | total=5314


In [ ]:
# 4. Entrenar en una sola línea
trained_model, history = train_network(model, train_loader, val_loader, optimizer, epochs=100, device=DEVICE, model_name="SegNet_2D", save_dir="/content/drive/MyDrive/TFM/resultados/SegNet_2D")

↻ Reanudando SegNet_2D desde la época 101 (última completada: 100)
=== ENTRENAMIENTO: SegNet_2D | cuda | AMP=True | aug_gpu=True | épocas 101-100 ===


In [ ]:
from segnet import SegNet
m = SegNet().to(DEVICE)
x = torch.randn(2, 3, 256, 256).to(DEVICE)
y = m(x)
print(y.shape)   # debe ser torch.Size([2, 1, 256, 256])
print(sum(p.numel() for p in m.parameters()))

In [ ]:
import subprocess, threading, time

def registrar_gpu(intervalo=15, archivo="/content/gpu_log.txt"):
    while True:
        out = subprocess.run(["nvidia-smi", "--query-gpu=utilization.gpu,memory.used",
                              "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
        with open(archivo, "a") as f:
            f.write(f"{time.strftime('%H:%M:%S')}  {out}\n")
        time.sleep(intervalo)

threading.Thread(target=registrar_gpu, daemon=True).start()
#prueba de GPU

In [ ]:
import importlib, utils_training
importlib.reload(utils_training)
from utils_training import train_network
!rm -f /content/drive/MyDrive/TFM/resultados/UNet_2D/*.pth /content/drive/MyDrive/TFM/resultados/UNet_2D/*.json
#Eliminación de datos anteriores

In [ ]:
!tail -n 30 /content/gpu_log.txt
#prueba de gpu